# 02 — Data Preparation

## Main question

**How is the raw FAA wildlife-strike dataset converted into a reproducible analytical dataset without introducing leakage or unsupported assumptions?**

This notebook consumes the dataset reviewed in Notebook 01 and creates the canonical cleaned artifacts used by later notebooks.

The workflow deliberately separates:

1. **Deterministic preparation**, which can be applied to the entire analytical period without learning from future records.
2. **Learned preprocessing**, such as imputation, scaling, encoding, rare-category thresholds, feature selection, and resampling. Those operations are defined later inside modelling pipelines and fitted only on training data.

The next notebook is `03_descriptive_analytics.ipynb`. It should load the full canonical analytical dataset created here rather than repeat cleaning rules.

## Approved preparation decisions

The following choices were agreed before consolidating this notebook.

| Topic | Decision | Reason and downstream effect |
|---|---|---|
| Analytical period | Retain 1990–2024 | This matches the approved project scope and keeps 2025 available as a later external temporal check. Partial 2026 records are not mixed into the main analytical population. |
| Repeated incident identifiers | Remove the redundant duplicate in each reviewed pair | Manual inspection found seven repeated identifiers whose paired rows matched except for synonymous squirrel labels. Removing one copy prevents double-counting. The removed rows are exported for audit. |
| Species labels | Preserve raw labels, standardize confirmed synonyms, and create broad wildlife groups | Exact labels remain useful for Notebook 03. The binary core model will use broad wildlife type and size rather than hundreds of exact species labels. Any frequency-based rare grouping must be learned from training data later. |
| Target conflicts | Preserve values and flag conflicts; exclude affected rows from severity analysis | Five records contain disagreement between binary damage and damage level. Their original values remain visible, but they are not trusted for conditional severity work. |
| Number seen/struck | Keep ordered categories, including Unknown | The source reports ranges rather than exact counts. Midpoints would create measurements that were never observed. |
| Missing values | Apply field-specific rules | Confirmed checkbox and count fields may use zero where blank means “none.” Other blanks remain distinct from unknown, not applicable, or historically unavailable values. |
| Extreme numerical values | Retain plausible extremes and flag them | Unusual height, speed, and distance values are not automatically errors. Impossible values are converted to missing only when a documented rule supports that treatment. |
| Airport coordinates | Do not restore during preparation; allow optional dashboard-only enrichment | The source fields are entirely empty and undocumented. A later interface may join a cited airport lookup for mapping, but those externally sourced values will remain outside the analytical and modelling feature sets. |
| Predictor design | Maintain core, extended, and airport-aware feature sets | This separates user-supplied scenario inputs from richer context and allows later testing of airport memorization. |
| Leakage protection | Export a full analytical file and a restricted binary-modelling file | Severity, component damage, injuries, costs, and operational consequences are retained for legitimate later analyses but cannot enter the binary predictor matrix. |
| Validation periods | Train 1990–2018, validate 2019–2021, test 2022–2024 | The chronological split supports future-year evaluation. A recent-window sensitivity analysis will be required later because reporting and damage prevalence change over time. |
| Learned preprocessing | Do not fit it here | Imputation, encoding, scaling, feature selection, rare-category thresholds, and resampling must be fitted inside training pipelines to prevent leakage. |

## 1. Setup and portable paths

The notebook searches common repository locations instead of relying on Google Drive or a machine-specific absolute path. Update only the candidate lists if the repository uses a different structure.

In [1]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

SEED = 42
np.random.seed(SEED)

ROOT = Path.cwd()
for parent in [ROOT, *ROOT.parents]:
    if (parent / "data").exists() or (parent / "docs").exists():
        ROOT = parent
        break

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
DOCS_DIR = ROOT / "docs"
OUTPUT_DIR = ROOT / "outputs" / "02_data_preparation"
MAPPING_DIR = DOCS_DIR / "mappings"

for folder in [PROCESSED_DIR, DOCS_DIR, OUTPUT_DIR, MAPPING_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)
print("Processed output:", PROCESSED_DIR)

Repository root: c:\Users\davis\Programing\MDA_programs\Capstone\faa-wildlife-strike-damage-analysis
Processed output: c:\Users\davis\Programing\MDA_programs\Capstone\faa-wildlife-strike-damage-analysis\data\processed


## 2. Load the source dataset

Notebook 01 established that the raw export contains **348,146 records and 103 fields**, covers 1990 through June 19, 2026, and has seven repeated `INDEX_NR` groups.

This cell prefers an upstream analytical export if one exists. Otherwise, it searches for the raw FAA CSV. The exact path used is recorded for reproducibility.

In [2]:
candidate_files = [
    PROCESSED_DIR / "faa_strikes_notebook01.csv",
    PROCESSED_DIR / "faa_strikes_understood.csv",
    RAW_DIR / "faa_wildlife_strikes.csv",
    RAW_DIR / "faa_strikes.csv",
    ROOT / "database" / "wildlife_strikes.csv",
    ROOT / "database" / "faa_wildlife_strikes.csv",
]

SOURCE_FILE = next((p for p in candidate_files if p.exists()), None)

if SOURCE_FILE is None:
    csv_candidates = [
        p for p in ROOT.rglob("*.csv")
        if "strike" in p.name.lower() and "clean" not in p.name.lower()
    ]
    SOURCE_FILE = csv_candidates[0] if csv_candidates else None

if SOURCE_FILE is None:
    raise FileNotFoundError(
        "No FAA strike CSV was found. Place the raw or Notebook 01 export in "
        "data/raw or data/processed, then rerun."
    )

# A utf-08 error appeard during testing
# likely from erroneous encoding from Access
# This section handles such error and 
# possible future correction.
try:
    df = pd.read_csv(
        SOURCE_FILE,
        encoding="utf-8",
        low_memory=False
    )
    file_encoding = "utf-8"

except UnicodeDecodeError:
    df = pd.read_csv(
        SOURCE_FILE,
        encoding="cp1252",
        low_memory=False
    )
    file_encoding = "cp1252"

print("Source file:", SOURCE_FILE)
print("Encoding used:", file_encoding)

Source file: c:\Users\davis\Programing\MDA_programs\Capstone\faa-wildlife-strike-damage-analysis\data\raw\faa_strikes.csv
Encoding used: cp1252


### Source validation

The checks below stop execution when the loaded file does not resemble the dataset audited in Notebook 01. Small row-count differences are allowed only after an upstream export has intentionally filtered the data.

In [3]:
required_fields = {
    "INDEX_NR", "INCIDENT_YEAR", "INCIDENT_MONTH", "INDICATED_DAMAGE",
    "DAMAGE_LEVEL", "SPECIES", "SIZE", "NUM_STRUCK"
}
missing_required = sorted(required_fields - set(df.columns))
if missing_required:
    raise ValueError(f"Required fields are missing: {missing_required}")

source_summary = pd.DataFrame({
    "measure": [
        "rows", "columns", "minimum year", "maximum year",
        "unique incident identifiers", "repeated identifier groups"
    ],
    "value": [
        len(df), df.shape[1],
        pd.to_numeric(df["INCIDENT_YEAR"], errors="coerce").min(),
        pd.to_numeric(df["INCIDENT_YEAR"], errors="coerce").max(),
        df["INDEX_NR"].nunique(dropna=True),
        int((df["INDEX_NR"].value_counts(dropna=False) > 1).sum())
    ]
})
display(source_summary)

,measure,value
0,rows,348146
1,columns,103
2,minimum year,1990
3,maximum year,2026
4,unique incident identifiers,348139
5,repeated identifier groups,7


## 3. Preserve raw values before standardization

Fields that are changed receive a companion `*_RAW` column. This makes mappings reversible and lets reviewers distinguish source values from project-standardized values.

In [4]:
raw_preserve_fields = [
    "SPECIES", "SPECIES_ID", "SIZE", "NUM_SEEN", "NUM_STRUCK",
    "PHASE_OF_FLIGHT", "TIME_OF_DAY", "WARNED", "SKY",
    "PRECIPITATION", "AC_CLASS", "DAMAGE_LEVEL"
]

for col in raw_preserve_fields:
    if col in df.columns and f"{col}_RAW" not in df.columns:
        df[f"{col}_RAW"] = df[col]

print("Raw companion fields created:", [
    f"{c}_RAW" for c in raw_preserve_fields if f"{c}_RAW" in df.columns
])

Raw companion fields created: ['SPECIES_RAW', 'SPECIES_ID_RAW', 'SIZE_RAW', 'NUM_SEEN_RAW', 'NUM_STRUCK_RAW', 'PHASE_OF_FLIGHT_RAW', 'TIME_OF_DAY_RAW', 'WARNED_RAW', 'SKY_RAW', 'PRECIPITATION_RAW', 'AC_CLASS_RAW', 'DAMAGE_LEVEL_RAW']


## 4. Define the analytical population

### Decision

The canonical analytical period is **1990–2024**.

This retains compatibility with the approved proposal and later notebooks. Records from 2025 and partial 2026 are not discarded permanently: they are exported separately so that 2025 can support an external temporal check after model development.

In [5]:
df["INCIDENT_YEAR"] = pd.to_numeric(df["INCIDENT_YEAR"], errors="coerce").astype("Int64")

excluded_period_records = df.loc[
    df["INCIDENT_YEAR"].isna() | ~df["INCIDENT_YEAR"].between(1990, 2024)
].copy()

analysis_df = df.loc[df["INCIDENT_YEAR"].between(1990, 2024)].copy()

excluded_period_records.to_csv(
    OUTPUT_DIR / "excluded_period_records_2025_2026_or_unknown.csv",
    index=False
)

print(f"Rows retained for 1990–2024: {len(analysis_df):,}")
print(f"Rows outside the analytical period: {len(excluded_period_records):,}")
display(
    excluded_period_records["INCIDENT_YEAR"]
    .value_counts(dropna=False)
    .sort_index()
    .rename("records")
)

Rows retained for 1990–2024: 319,111
Rows outside the analytical period: 29,035


INCIDENT_YEAR
2025    24459
2026     4576
Name: records, dtype: Int64

## 5. Review and resolve repeated incident identifiers

Notebook 01 found no exact duplicate rows but identified seven repeated `INDEX_NR` groups. Manual review showed that each reviewed pair describes the same event and differs because one record uses **“Squirrels”** while the other uses **“Tree squirrels.”**

### Decision

1. Map both labels to the canonical label `Tree squirrels`.
2. Re-evaluate duplicates after the mapping.
3. Keep one record per now-identical pair.
4. Export every removed row and the comparison evidence.

This prevents seven events from being counted twice while preserving a transparent audit trail.

In [6]:
SPECIES_SYNONYM_MAP = {
    "Squirrels": "Tree squirrels",
    "Tree Squirrels": "Tree squirrels",
    "TREE SQUIRRELS": "Tree squirrels",
}

species_mapping = pd.DataFrame(
    [{"raw_value": k, "canonical_value": v, "reason": "Reviewed synonym"}
     for k, v in SPECIES_SYNONYM_MAP.items()]
)
species_mapping.to_csv(MAPPING_DIR / "species_synonym_mapping.csv", index=False)

analysis_df["SPECIES"] = (
    analysis_df["SPECIES"]
    .astype("string")
    .str.strip()
    .replace(SPECIES_SYNONYM_MAP)
)

repeated_before = analysis_df[
    analysis_df["INDEX_NR"].duplicated(keep=False)
].sort_values("INDEX_NR").copy()

comparison_excluded = {
    "SPECIES_RAW", "SPECIES"
}
comparison_columns = [
    c for c in analysis_df.columns
    if c not in comparison_excluded
]

drop_indices = []
review_rows = []

for incident_id, group in repeated_before.groupby("INDEX_NR", dropna=False):
    identical_except_species = (
        len(group) > 1 and
        group[comparison_columns].nunique(dropna=False).le(1).all() and
        group["SPECIES"].nunique(dropna=False) == 1
    )
    review_rows.append({
        "INDEX_NR": incident_id,
        "rows": len(group),
        "identical_after_species_mapping": bool(identical_except_species),
        "raw_species_labels": " | ".join(
            sorted(group["SPECIES_RAW"].astype(str).unique())
        )
    })
    if identical_except_species:
        drop_indices.extend(group.index[1:].tolist())

duplicate_review = pd.DataFrame(review_rows)
removed_repeated_rows = analysis_df.loc[drop_indices].copy()

duplicate_review.to_csv(OUTPUT_DIR / "repeated_identifier_decision.csv", index=False)
removed_repeated_rows.to_csv(OUTPUT_DIR / "removed_redundant_records.csv", index=False)

analysis_df = analysis_df.drop(index=drop_indices).copy()

unresolved_repeated = analysis_df[
    analysis_df["INDEX_NR"].duplicated(keep=False)
].sort_values("INDEX_NR")

print("Repeated groups before rule:", repeated_before["INDEX_NR"].nunique())
print("Rows removed:", len(removed_repeated_rows))
print("Repeated groups still unresolved:", unresolved_repeated["INDEX_NR"].nunique())
display(duplicate_review)

Repeated groups before rule: 4
Rows removed: 4
Repeated groups still unresolved: 0


,INDEX_NR,rows,identical_after_species_mapping,raw_species_labels
0,778860,2,True,Squirrels | Tree squirrels
1,793166,2,True,Squirrels | Tree squirrels
2,814615,2,True,Squirrels | Tree squirrels
3,1520761,2,True,Squirrels | Tree squirrels


### Validation rule

The expected result is seven removed rows and no unresolved repeated identifiers. If other repeated groups appear after the source data is updated, the notebook does **not** delete them automatically; they must be reviewed.

In [7]:
if len(removed_repeated_rows) != 7:
    warnings.warn(
        f"Expected 7 redundant rows from the reviewed source, but removed "
        f"{len(removed_repeated_rows)}. Review the exported comparison table."
    )

if not unresolved_repeated.empty:
    warnings.warn(
        "Some repeated INDEX_NR groups remain unresolved. They were preserved "
        "and exported for manual review."
    )
    unresolved_repeated.to_csv(
        OUTPUT_DIR / "unresolved_repeated_identifiers.csv", index=False
    )

C:\Users\davis\AppData\Local\Temp\ipykernel_33528\864592507.py:2: UserWarning: Expected 7 redundant rows from the reviewed source, but removed 4. Review the exported comparison table.
  warnings.warn(


## 6. Standardize missing categorical values without erasing meaning

Missingness is not handled with one universal label.

- Existing source categories such as `Unknown` remain `Unknown`.
- Blank values become `Not reported` when the field should normally contain an observation but the source provides no more specific explanation.
- `Not applicable` is used only where a documented structural rule supports it.
- Historically unavailable values should remain distinguishable when a year-based rule can be verified.

This distinction matters because damaged incidents are often more completely documented than non-damaged incidents. Missingness may therefore contain reporting information rather than random noise.

In [8]:
categorical_fields = [
    "SIZE", "NUM_SEEN", "NUM_STRUCK", "WARNED", "SKY",
    "PRECIPITATION", "TIME_OF_DAY", "PHASE_OF_FLIGHT",
    "AC_CLASS", "SPECIES"
]

def normalize_text(series):
    out = series.astype("string").str.strip()
    out = out.replace({
        "": pd.NA,
        "nan": pd.NA,
        "NaN": pd.NA,
        "NAN": pd.NA
    })
    return out

for col in categorical_fields:
    if col in analysis_df.columns:
        analysis_df[col] = normalize_text(analysis_df[col])

# SPECIES was complete in Notebook 01, but the fallback remains explicit.
analysis_df["SPECIES"] = analysis_df["SPECIES"].fillna("Unknown wildlife")

# For these observational categories, a blank does not reveal whether the
# reporter did not know, did not observe, or omitted the answer.
for col in [
    "SIZE", "NUM_SEEN", "NUM_STRUCK", "SKY", "PRECIPITATION",
    "TIME_OF_DAY", "PHASE_OF_FLIGHT", "AC_CLASS"
]:
    if col in analysis_df.columns:
        analysis_df[col] = analysis_df[col].fillna("Not reported")

# WARNED has an explicit Unknown source category; blanks remain distinguishable.
if "WARNED" in analysis_df.columns:
    analysis_df["WARNED"] = analysis_df["WARNED"].fillna("Not reported")

category_missing_summary = pd.DataFrame([
    {
        "field": col,
        "not_reported_count": int((analysis_df[col] == "Not reported").sum()),
        "unknown_count": int(
            analysis_df[col].astype(str).str.contains("unknown", case=False, na=False).sum()
        ),
        "distinct_values": int(analysis_df[col].nunique(dropna=False))
    }
    for col in categorical_fields if col in analysis_df.columns
])
display(category_missing_summary)

,field,not_reported_count,unknown_count,distinct_values
0,SIZE,31492,0,4
1,NUM_SEEN,215986,0,5
2,NUM_STRUCK,681,0,5
3,WARNED,0,198092,3
4,SKY,167782,0,4
5,PRECIPITATION,307138,0,11
6,TIME_OF_DAY,138648,0,5
7,PHASE_OF_FLIGHT,124521,0,12
8,AC_CLASS,90825,0,6
9,SPECIES,0,126565,927


## 7. Standardize `NUM_SEEN` and `NUM_STRUCK` as ordered categories

These variables are reported as categories or ranges, not exact continuous measurements. Spreadsheet software may display `2–10` as a date-like value such as `10-Feb`.

### Decision

The cleaned values are:

1. `1`
2. `2–10`
3. `11–100`
4. `More than 100`
5. `Unknown`
6. `Not reported`

Unknown and unreported cases are retained because uncertainty may itself describe how the event was observed or documented.

In [9]:
COUNT_CATEGORY_MAP = {
    "1": "1",
    "10-Feb": "2–10",
    "Feb-10": "2–10",
    "2-10": "2–10",
    "2 – 10": "2–10",
    "2 to 10": "2–10",
    "11-100": "11–100",
    "11 – 100": "11–100",
    "11 to 100": "11–100",
    "More than 100": "More than 100",
    ">100": "More than 100",
    "Unknown": "Unknown",
    "UNKNOWN": "Unknown",
    "Not reported": "Not reported",
}

count_mapping_rows = []

for col in ["NUM_SEEN", "NUM_STRUCK"]:
    if col not in analysis_df.columns:
        continue

    before = analysis_df[col].copy()
    analysis_df[col] = before.replace(COUNT_CATEGORY_MAP)

    for raw_value, canonical_value in (
        pd.DataFrame({"raw": before, "canonical": analysis_df[col]})
        .drop_duplicates()
        .sort_values(["canonical", "raw"], na_position="last")
        .itertuples(index=False)
    ):
        count_mapping_rows.append({
            "field": col,
            "raw_value": raw_value,
            "canonical_value": canonical_value
        })

count_category_order = [
    "1", "2–10", "11–100", "More than 100", "Unknown", "Not reported"
]

for col in ["NUM_SEEN", "NUM_STRUCK"]:
    if col in analysis_df.columns:
        unexpected = sorted(
            set(analysis_df[col].dropna().unique()) - set(count_category_order)
        )
        if unexpected:
            warnings.warn(f"Unexpected {col} categories preserved for review: {unexpected}")
        analysis_df[f"{col}_ORDER"] = analysis_df[col].map({
            "1": 1, "2–10": 2, "11–100": 3, "More than 100": 4
        }).astype("Int64")

pd.DataFrame(count_mapping_rows).to_csv(
    MAPPING_DIR / "count_category_mapping.csv", index=False
)

display(analysis_df["NUM_STRUCK"].value_counts(dropna=False).rename("records"))

NUM_STRUCK
1                283177
2–10              33597
11–100             1595
Not reported        681
More than 100        57
Name: records, dtype: Int64

## 8. Target validation and conflict handling

`INDICATED_DAMAGE` is the primary binary target.

`DAMAGE_LEVEL` is a conditional severity field with the following meanings:

- Blank: unknown
- `N`: no reported damage
- `M`: minor damage
- `M?`: damaged, but the extent is undetermined
- `S`: substantial damage
- `D`: destroyed

### Decision

The original target values are not overwritten. A conflict flag identifies records where:

- `INDICATED_DAMAGE = 1` but `DAMAGE_LEVEL = N`, or
- `INDICATED_DAMAGE = 0` but `DAMAGE_LEVEL` indicates damage.

These records remain available for the binary analysis, but later severity analysis must exclude them unless a source-based correction is established.

In [10]:
analysis_df["INDICATED_DAMAGE"] = pd.to_numeric(
    analysis_df["INDICATED_DAMAGE"], errors="coerce"
).astype("Int64")

analysis_df["DAMAGE_LEVEL"] = normalize_text(
    analysis_df["DAMAGE_LEVEL"]
)

damage_level_map = {
    "N": "N",
    "M": "M",
    "M?": "M?",
    "S": "S",
    "D": "D"
}
analysis_df["DAMAGE_LEVEL"] = analysis_df["DAMAGE_LEVEL"].replace(damage_level_map)

damage_codes = {"M", "M?", "S", "D"}

analysis_df["TARGET_CONFLICT_FLAG"] = (
    (
        analysis_df["INDICATED_DAMAGE"].eq(1) &
        analysis_df["DAMAGE_LEVEL"].eq("N")
    ) |
    (
        analysis_df["INDICATED_DAMAGE"].eq(0) &
        analysis_df["DAMAGE_LEVEL"].isin(damage_codes)
    )
).astype("Int64")

analysis_df["SEVERITY_ELIGIBLE_FLAG"] = (
    analysis_df["INDICATED_DAMAGE"].eq(1) &
    analysis_df["DAMAGE_LEVEL"].isin(damage_codes) &
    analysis_df["TARGET_CONFLICT_FLAG"].eq(0)
).astype("Int64")

target_conflicts = analysis_df.loc[
    analysis_df["TARGET_CONFLICT_FLAG"].eq(1),
    ["INDEX_NR", "INDICATED_DAMAGE", "DAMAGE_LEVEL",
     "INDICATED_DAMAGE_RAW" if "INDICATED_DAMAGE_RAW" in analysis_df.columns else "INDICATED_DAMAGE"]
].copy()

target_conflicts.to_csv(OUTPUT_DIR / "target_conflicts.csv", index=False)

print("Primary-target missing values:", analysis_df["INDICATED_DAMAGE"].isna().sum())
print("Target conflicts:", int(analysis_df["TARGET_CONFLICT_FLAG"].sum()))
display(
    analysis_df.groupby(
        ["INDICATED_DAMAGE", "DAMAGE_LEVEL"], dropna=False
    ).size().rename("records").reset_index()
)

Primary-target missing values: 0
Target conflicts: 0


,INDICATED_DAMAGE,DAMAGE_LEVEL,records
0,0,N,185386
1,0,<NA>,112813
2,1,D,89
3,1,M,8657
4,1,M?,7818
5,1,S,4344


## 9. Field-specific zero conversion

A low or high missing percentage does not determine whether zero is valid. The meaning of the blank does.

### Applied rule

- `STR_*`, `DAM_*`, and `ING_*` fields are treated as checkbox indicators only after confirming that their observed non-null values are binary.
- Blank injury and fatality counts are converted to zero because these fields record counts of reported outcomes.
- Administrative indicators such as `IMAGE` and remains fields are **not** automatically converted to zero. A blank may mean unreported rather than “No.”
- `INDICATED_DAMAGE` is never blanket-filled because it is the target.

The before-and-after table is exported so that every conversion can be reviewed.

In [11]:
zero_treatment_rows = []

checkbox_fields = [
    c for c in analysis_df.columns
    if c.startswith(("STR_", "DAM_", "ING_"))
    and not c.endswith("_RAW")
]

for col in checkbox_fields:
    numeric = pd.to_numeric(analysis_df[col], errors="coerce")
    observed = set(numeric.dropna().unique())

    if observed.issubset({0, 1}):
        missing_before = int(numeric.isna().sum())
        analysis_df[col] = numeric.fillna(0).astype("Int64")
        zero_treatment_rows.append({
            "field": col,
            "missing_before": missing_before,
            "action": "Blank to 0",
            "reason": "Verified binary checkbox indicator"
        })
    else:
        zero_treatment_rows.append({
            "field": col,
            "missing_before": int(analysis_df[col].isna().sum()),
            "action": "Preserved",
            "reason": f"Observed values were not strictly binary: {sorted(observed)}"
        })

for col in ["NR_INJURIES", "NR_FATALITIES"]:
    if col in analysis_df.columns:
        numeric = pd.to_numeric(analysis_df[col], errors="coerce")
        missing_before = int(numeric.isna().sum())
        analysis_df[col] = numeric.fillna(0)
        zero_treatment_rows.append({
            "field": col,
            "missing_before": missing_before,
            "action": "Blank to 0",
            "reason": "Reported outcome count; blank interpreted as no reported cases"
        })

for col in ["IMAGE", "REMAINS_COLLECTED", "REMAINS_SENT", "INGESTED_OTHER"]:
    if col in analysis_df.columns:
        zero_treatment_rows.append({
            "field": col,
            "missing_before": int(analysis_df[col].isna().sum()),
            "action": "Preserved",
            "reason": "Administrative or reporting blank is not proven to mean No"
        })

zero_treatment_report = pd.DataFrame(zero_treatment_rows)
zero_treatment_report.to_csv(
    OUTPUT_DIR / "zero_conversion_decisions.csv", index=False
)
display(zero_treatment_report)

,field,missing_before,action,reason
0,STR_RAD,0,Blank to 0,Verified binary checkbox indicator
1,DAM_RAD,0,Blank to 0,Verified binary checkbox indicator
2,STR_WINDSHLD,0,Blank to 0,Verified binary checkbox indicator
3,DAM_WINDSHLD,0,Blank to 0,Verified binary checkbox indicator
4,STR_NOSE,0,Blank to 0,Verified binary checkbox indicator
5,DAM_NOSE,0,Blank to 0,Verified binary checkbox indicator
6,STR_ENG1,0,Blank to 0,Verified binary checkbox indicator
7,DAM_ENG1,0,Blank to 0,Verified binary checkbox indicator
8,ING_ENG1,0,Blank to 0,Verified binary checkbox indicator
9,STR_ENG2,0,Blank to 0,Verified binary checkbox indicator


## 10. Broad wildlife groups

Exact species labels may suggest meaningful differences, but using 900+ labels directly in the core model would create sparse categories and weak support for many scenarios.

### Decision

- Preserve standardized `SPECIES` for descriptive analysis.
- Create a transparent broad `WILDLIFE_TYPE` feature.
- Use `WILDLIFE_TYPE` and `SIZE` in the core model.
- Keep exact species outside the core predictor set.
- Any later rare-category threshold for exact species must be learned from the training period only.

The grouping is intentionally broad. It does not claim biological taxonomy beyond what the text label supports.

In [12]:
def broad_wildlife_type(label):
    if pd.isna(label):
        return "Unknown wildlife"

    text = str(label).lower()

    if "bat" in text:
        return "Bat"
    if any(term in text for term in [
        "squirrel", "rabbit", "hare", "deer", "coyote", "fox",
        "raccoon", "opossum", "skunk", "dog", "cat", "mammal",
        "rodent", "woodchuck", "groundhog"
    ]):
        return "Terrestrial mammal"
    if any(term in text for term in [
        "turtle", "tortoise", "snake", "alligator", "crocodile",
        "lizard", "reptile"
    ]):
        return "Reptile"
    if any(term in text for term in ["frog", "toad", "amphibian"]):
        return "Amphibian"
    if any(term in text for term in [
        "bird", "gull", "goose", "geese", "duck", "hawk", "eagle",
        "owl", "sparrow", "swallow", "pigeon", "dove", "heron",
        "egret", "turkey", "vulture", "crane", "pelican", "tern",
        "blackbird", "starling", "lark", "falcon", "kite", "sandpiper"
    ]):
        return "Bird"
    if "unknown" in text:
        return "Unknown wildlife"
    return "Other wildlife"

analysis_df["WILDLIFE_TYPE"] = analysis_df["SPECIES"].map(
    broad_wildlife_type
)

wildlife_type_summary = (
    analysis_df.groupby("WILDLIFE_TYPE", dropna=False)
    .agg(
        records=("INDEX_NR", "size"),
        damage_rate=("INDICATED_DAMAGE", "mean"),
        distinct_species=("SPECIES", "nunique")
    )
    .sort_values("records", ascending=False)
)
wildlife_type_summary["damage_rate"] *= 100

display(wildlife_type_summary.round(2))

,records,damage_rate,distinct_species
WILDLIFE_TYPE,,,
Bird,252068,6.78,308
Other wildlife,40600,5.53,466
Terrestrial mammal,19339,7.74,71
Bat,6427,0.98,44
Reptile,673,0.59,38


### Manual mapping review

The keyword grouping above is reproducible but should be peer-reviewed. The export below lists every exact label, its assigned broad group, and its frequency. Corrections should be made through a mapping table rather than editing individual records.

In [13]:
species_group_review = (
    analysis_df.groupby(["SPECIES", "WILDLIFE_TYPE"], dropna=False)
    .size()
    .rename("records")
    .reset_index()
    .sort_values("records", ascending=False)
)

species_group_review.to_csv(
    MAPPING_DIR / "species_to_wildlife_type_review.csv", index=False
)
display(species_group_review.head(30))

,SPECIES,WILDLIFE_TYPE,records
809,Unknown bird - small,Bird,52259
808,Unknown bird - medium,Bird,38955
806,Unknown bird,Bird,31049
519,Mourning dove,Bird,16374
52,Barn swallow,Bird,10838
429,Killdeer,Terrestrial mammal,10663
21,American kestrel,Other wildlife,9791
406,Horned lark,Bird,9209
378,Gulls,Bird,7551
297,European starling,Bird,6622


## 11. Aircraft, flight, weather, warning, temporal, and geographic categories

Only deterministic text normalization is performed here. Categories are not merged solely because they are rare. Any model-dependent category reduction belongs inside the training pipeline.

In [14]:
standardize_fields = [
    "AC_CLASS", "TYPE_ENG", "PHASE_OF_FLIGHT", "TIME_OF_DAY",
    "WARNED", "SKY", "PRECIPITATION", "STATE", "FAAREGION",
    "AIRPORT_ID", "AIRPORT"
]

category_mapping_frames = []

for col in standardize_fields:
    if col not in analysis_df.columns:
        continue

    raw = analysis_df[col].astype("string")
    cleaned = raw.str.strip().replace({"": pd.NA})
    cleaned = cleaned.fillna("Not reported")

    mapping = pd.DataFrame({
        "field": col,
        "raw_value": raw,
        "canonical_value": cleaned
    }).drop_duplicates()

    category_mapping_frames.append(mapping)
    analysis_df[col] = cleaned

category_mapping_table = pd.concat(
    category_mapping_frames, ignore_index=True
) if category_mapping_frames else pd.DataFrame()

category_mapping_table.to_csv(
    MAPPING_DIR / "general_category_mapping.csv", index=False
)

display(
    pd.DataFrame({
        "field": [c for c in standardize_fields if c in analysis_df.columns],
        "distinct_values": [
            analysis_df[c].nunique(dropna=False)
            for c in standardize_fields if c in analysis_df.columns
        ]
    }).sort_values("distinct_values", ascending=False)
)

,field,distinct_values
9,AIRPORT_ID,2708
10,AIRPORT,2708
7,STATE,68
2,PHASE_OF_FLIGHT,12
6,PRECIPITATION,11
8,FAAREGION,11
1,TYPE_ENG,8
0,AC_CLASS,6
3,TIME_OF_DAY,5
5,SKY,4


### Airport-coordinate fields and possible dashboard enrichment

The source Access database contains `AIRPORT_LATITUDE` and
`AIRPORT_LONGITUDE`, but both fields are completely empty. They are also
absent from the supplied official data dictionary, so their intended
definition and provenance cannot be verified. No values are imputed or
externally added during preparation.

Airport geography in the analytical workflow is represented through
`AIRPORT_ID`, `AIRPORT`, `STATE`, and `FAAREGION`. These fields are sufficient
for airport-level descriptions, geographic comparisons, and airport-held-out
validation.

Coordinates may be added later as a separate dashboard-only enrichment if an
airport map is included. In that case, the interface should load a cited
airport reference table and join coordinates through a validated
`AIRPORT_ID` match. The added values must be clearly labelled as externally
sourced, used only for visualization, and kept outside the model feature
lists. This avoids presenting externally obtained airport locations as
observations from the original wildlife-strike dataset.

## 12. Numerical types, units, and plausibility flags

Notebook 01 found:

- no negative `HEIGHT`, `SPEED`, or `DISTANCE` values,
- a maximum height of 40,000 ft,
- a maximum speed of 541,
- a maximum distance of 99,
- no Taxi/Parked records with height above 50 ft.

Extreme values are therefore retained rather than winsorized. The notebook adds review flags instead of silently changing unusual observations.

Negative values, should they appear in a refreshed source, are converted to missing because the reviewed FAA definitions do not support negative height, speed, or distance.

In [15]:
numeric_fields = [
    "HEIGHT", "SPEED", "DISTANCE", "NUM_ENGS", "AC_MASS",
    "NR_INJURIES", "NR_FATALITIES"
]

numeric_review_rows = []

for col in numeric_fields:
    if col not in analysis_df.columns:
        continue

    analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce")
    negative_mask = analysis_df[col] < 0
    negative_count = int(negative_mask.sum())

    analysis_df[f"{col}_INVALID_NEGATIVE_FLAG"] = negative_mask.astype("Int64")
    analysis_df.loc[negative_mask, col] = np.nan

    q999 = analysis_df[col].quantile(0.999)
    analysis_df[f"{col}_EXTREME_REVIEW_FLAG"] = (
        analysis_df[col] > q999
    ).astype("Int64")

    numeric_review_rows.append({
        "field": col,
        "missing_pct": round(analysis_df[col].isna().mean() * 100, 2),
        "negative_values_converted_to_missing": negative_count,
        "minimum": analysis_df[col].min(),
        "p99_9": q999,
        "maximum": analysis_df[col].max(),
        "records_above_p99_9": int(
            analysis_df[f"{col}_EXTREME_REVIEW_FLAG"].sum()
        )
    })

if {"PHASE_OF_FLIGHT", "HEIGHT"}.issubset(analysis_df.columns):
    analysis_df["GROUND_PHASE_HEIGHT_CONFLICT_FLAG"] = (
        analysis_df["PHASE_OF_FLIGHT"].isin(["Taxi", "Parked"]) &
        analysis_df["HEIGHT"].gt(50)
    ).astype("Int64")

numeric_review = pd.DataFrame(numeric_review_rows)
numeric_review.to_csv(
    OUTPUT_DIR / "numeric_plausibility_review.csv", index=False
)
display(numeric_review)

,field,missing_pct,negative_values_converted_to_missing,minimum,p99_9,maximum,records_above_p99_9
0,HEIGHT,49.57,0,0.0,14500.0,32000.0,158
1,SPEED,68.42,0,0.0,310.0,541.0,90
2,DISTANCE,33.55,0,0.0,40.0,99.0,172
3,NUM_ENGS,28.58,0,1.0,4.0,4.0,0
4,AC_MASS,28.50,0,1.0,5.0,5.0,0
5,NR_INJURIES,0.00,0,0.0,0.0,7.0,294
6,NR_FATALITIES,0.00,0,0.0,0.0,8.0,25


## 13. Deterministic feature engineering

The engineered features below can be reproduced without estimating parameters from the data.

- `SEASON` is derived from calendar month.
- `MONTH_SIN` and `MONTH_COS` preserve the cyclical relationship between December and January.
- `WILDLIFE_TYPE` is generated from the saved species mapping logic.
- `AC_MASS_GROUP` collapses official aircraft-mass codes into broad groups for scenario communication.
- Missingness indicators are created only for fields where absence may reflect reporting differences.

No imputation, standardization, one-hot encoding, feature selection, or resampling is fitted here.

In [16]:
analysis_df["INCIDENT_MONTH"] = pd.to_numeric(
    analysis_df["INCIDENT_MONTH"], errors="coerce"
).astype("Int64")

def month_to_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    if month in [3, 4, 5]:
        return "Spring"
    if month in [6, 7, 8]:
        return "Summer"
    if month in [9, 10, 11]:
        return "Autumn"
    return "Unknown"

analysis_df["SEASON"] = analysis_df["INCIDENT_MONTH"].map(month_to_season)

month_numeric = analysis_df["INCIDENT_MONTH"].astype("float")
analysis_df["MONTH_SIN"] = np.sin(2 * np.pi * month_numeric / 12)
analysis_df["MONTH_COS"] = np.cos(2 * np.pi * month_numeric / 12)

mass_group_map = {
    1: "Light",
    2: "Light",
    3: "Medium",
    4: "Heavy",
    5: "Heavy"
}
if "AC_MASS" in analysis_df.columns:
    analysis_df["AC_MASS_GROUP"] = (
        pd.to_numeric(analysis_df["AC_MASS"], errors="coerce")
        .map(mass_group_map)
        .fillna("Unknown")
    )

missing_indicator_fields = [
    "HEIGHT", "SPEED", "TIME_OF_DAY", "PHASE_OF_FLIGHT",
    "SIZE", "SKY", "PRECIPITATION"
]
for col in missing_indicator_fields:
    if col in analysis_df.columns:
        if pd.api.types.is_numeric_dtype(analysis_df[col]):
            analysis_df[f"{col}_MISSING_FLAG"] = analysis_df[col].isna().astype("Int64")
        else:
            analysis_df[f"{col}_MISSING_FLAG"] = (
                analysis_df[col].isin(["Unknown", "Not reported"])
            ).astype("Int64")

display(
    analysis_df[[
        "INCIDENT_MONTH", "SEASON", "MONTH_SIN", "MONTH_COS",
        "WILDLIFE_TYPE", "AC_MASS_GROUP"
    ]].head()
)

,INCIDENT_MONTH,SEASON,MONTH_SIN,MONTH_COS,WILDLIFE_TYPE,AC_MASS_GROUP
0,6,Summer,1.224647e-16,-1.000000,Bird,Heavy
1,6,Summer,1.224647e-16,-1.000000,Bird,Heavy
2,7,Summer,-5.000000e-01,-0.866025,Bird,Heavy
3,7,Summer,-5.000000e-01,-0.866025,Bird,Heavy
4,7,Summer,-5.000000e-01,-0.866025,Bird,Heavy


## 14. Canonical feature-eligibility and leakage table

The table is the source of truth for every downstream model.

### Predictor-set logic

**Core features** represent information that can reasonably describe a user-selected or pre-impact scenario.

**Extended features** add richer contextual variables that may be incomplete or unavailable in some interface scenarios.

**Airport-aware features** add airport identity for comparison. This version tests whether predictive performance comes partly from memorizing airports and is not automatically selected as the final model.

Exact `SPECIES` is preserved for Notebook 03 but excluded from the core model. Later modelling may test it only as an extended, training-controlled feature.

In [ ]:
PRIMARY_TARGET = "INDICATED_DAMAGE"
SECONDARY_TARGETS = ["DAMAGE_LEVEL"]
COMPONENT_TARGETS = [
    c for c in analysis_df.columns
    if c.startswith("DAM_") and not c.endswith(("_RAW", "_FLAG"))
]

CORE_FEATURES = [
    c for c in [
        "SEASON", "MONTH_SIN", "MONTH_COS",
        "WILDLIFE_TYPE", "SIZE", "NUM_STRUCK",
        "AC_CLASS", "AC_MASS_GROUP", "TYPE_ENG", "NUM_ENGS",
        "WARNED"
    ] if c in analysis_df.columns
]

EXTENDED_FEATURES = CORE_FEATURES + [
    c for c in [
        "PHASE_OF_FLIGHT", "HEIGHT", "SPEED", "TIME_OF_DAY",
        "SKY", "PRECIPITATION", "STATE", "FAAREGION"
    ] if c in analysis_df.columns and c not in CORE_FEATURES
]

AIRPORT_AWARE_FEATURES = EXTENDED_FEATURES + [
    c for c in ["AIRPORT_ID"]
    if c in analysis_df.columns and c not in EXTENDED_FEATURES
]

identifiers = {
    c for c in ["INDEX_NR", "REG", "FLT", "REPORTED_NAME"]
    if c in analysis_df.columns
}

administrative = {
    c for c in [
        "IMAGE", "SOURCE", "PERSON", "LUPDATE", "TRANSFER",
        "REPORTED_TITLE", "COMMENTS", "REMARKS"
    ] if c in analysis_df.columns
}

post_event = {
    c for c in [
        "DAMAGE_LEVEL", "EFFECT", "EFFECT_OTHER", "AOS",
        "COST_REPAIRS", "COST_OTHER", "COST_REPAIRS_INFL_ADJ",
        "COST_OTHER_INFL_ADJ", "NR_INJURIES", "NR_FATALITIES"
    ] if c in analysis_df.columns
}

post_event |= {
    c for c in analysis_df.columns
    if c.startswith(("STR_", "DAM_", "ING_"))
    and not c.endswith(("_RAW", "_FLAG"))
}

post_event |= {
    c for c in [
        "INGESTED_OTHER", "REMAINS_COLLECTED", "REMAINS_SENT",
        "BIRD_BAND_NUMBER", "OTHER_SPECIFY"
    ] if c in analysis_df.columns
}

descriptive_only = {
    c for c in [
        "SPECIES", "SPECIES_ID", "AIRPORT", "RUNWAY",
        "OPID", "OPERATOR", "AMA", "AMO", "EMA", "EMO",
        "LOCATION", "ENROUTE_STATE", "DISTANCE"
    ] if c in analysis_df.columns
}

undocumented_empty_fields = {
    c for c in [
        "AIRPORT_LATITUDE",
        "AIRPORT_LONGITUDE",
        "INCIDENT_LATITUDE",
        "INCIDENT_LONGITUDE"
    ]
    if c in analysis_df.columns and analysis_df[c].isna().all()
}

raw_companions = {c for c in analysis_df.columns if c.endswith("_RAW")}

def classify_field(field):
    if field == PRIMARY_TARGET:
        return ("primary_target", "Target only", "Primary binary outcome")
    if field in SECONDARY_TARGETS:
        return ("secondary_target", "Exclude", "Post-event severity target")
    if field in COMPONENT_TARGETS:
        return ("component_target", "Exclude", "Post-event component outcome")
    if field in identifiers:
        return ("identifier", "Exclude", "Identifier; not a scenario predictor")
    if field in administrative:
        return ("administrative", "Exclude", "Reporting or administrative field")
    if field in post_event:
        return ("excluded_leakage", "Exclude", "Known after impact or consequence")
    if field in raw_companions:
        return ("raw_audit_field", "Exclude", "Preserved source value for audit")
    if field in CORE_FEATURES:
        return ("core_feature", "Include", "Scenario-eligible core predictor")
    if field in AIRPORT_AWARE_FEATURES and field not in EXTENDED_FEATURES:
        return ("airport_feature", "Airport-aware only", "Tests airport dependence")
    if field in EXTENDED_FEATURES:
        return ("extended_feature", "Extended", "Contextual predictor")
    if field in undocumented_empty_fields:
        return ("undocumented_empty_field","Exclude", "Entirely empty and undocumented; optional external coordinates are reserved for dashboard mapping")
    if field in descriptive_only:
        return ("descriptive_only", "Exclude", "Retained for historical description")
    return ("pending_review", "Exclude by default", "Requires documented approval")

eligibility_rows = []
for field in analysis_df.columns:
    role, binary_use, rationale = classify_field(field)
    eligibility_rows.append({
        "field": field,
        "role": role,
        "available_before_or_at_scenario": (
            "Yes" if role in {"core_feature", "extended_feature", "airport_feature"}
            else "No or not applicable"
        ),
        "leakage_risk": (
            "High" if role in {
                "secondary_target", "component_target", "excluded_leakage"
            } else "Low/controlled" if role in {
                "core_feature", "extended_feature", "airport_feature"
            } else "Not used"
        ),
        "binary_damage_use": binary_use,
        "severity_use": (
            "Target" if field == "DAMAGE_LEVEL"
            else "Candidate" if role in {"core_feature", "extended_feature"}
            else "Exclude"
        ),
        "component_use": (
            "Target" if field in COMPONENT_TARGETS
            else "Candidate" if role in {"core_feature", "extended_feature"}
            else "Exclude"
        ),
        "rationale": rationale,
        "review_status": "Approved in Notebook 02" if role != "pending_review" else "Manual review required"
    })

feature_eligibility = pd.DataFrame(eligibility_rows)
feature_eligibility.to_csv(
    DOCS_DIR / "feature_eligibility.csv", index=False
)

display(feature_eligibility["role"].value_counts())
display(feature_eligibility.head(25))

role
pending_review      39
excluded_leakage    32
descriptive_only    15
component_target    14
raw_audit_field     12
core_feature        11
extended_feature     8
administrative       8
identifier           4
airport_feature      1
primary_target       1
secondary_target     1
Name: count, dtype: int64

,field,role,available_before_or_at_scenario,leakage_risk,binary_damage_use,severity_use,component_use,rationale,review_status
0,INDEX_NR,identifier,No or not applicable,Not used,Exclude,Exclude,Exclude,Identifier; not a scenario predictor,Approved in Notebook 02
1,INCIDENT_DATE,pending_review,No or not applicable,Not used,Exclude by default,Exclude,Exclude,Requires documented approval,Manual review required
2,INCIDENT_MONTH,pending_review,No or not applicable,Not used,Exclude by default,Exclude,Exclude,Requires documented approval,Manual review required
3,INCIDENT_YEAR,pending_review,No or not applicable,Not used,Exclude by default,Exclude,Exclude,Requires documented approval,Manual review required
4,TIME,pending_review,No or not applicable,Not used,Exclude by default,Exclude,Exclude,Requires documented approval,Manual review required
5,TIME_OF_DAY,extended_feature,Yes,Low/controlled,Extended,Candidate,Candidate,Contextual predictor,Approved in Notebook 02
6,AIRPORT_ID,airport_feature,Yes,Low/controlled,Airport-aware only,Exclude,Exclude,Tests airport dependence,Approved in Notebook 02
7,AIRPORT,descriptive_only,No or not applicable,Not used,Exclude,Exclude,Exclude,Retained for historical description,Approved in Notebook 02
8,AIRPORT_LATITUDE,pending_review,No or not applicable,Not used,Exclude by default,Exclude,Exclude,Requires documented approval,Manual review required
9,AIRPORT_LONGITUDE,pending_review,No or not applicable,Not used,Exclude by default,Exclude,Exclude,Requires documented approval,Manual review required


### Feature-list artifact

In [18]:
feature_lists = {
    "primary_target": PRIMARY_TARGET,
    "secondary_targets": SECONDARY_TARGETS,
    "component_targets": COMPONENT_TARGETS,
    "core_features": CORE_FEATURES,
    "extended_features": EXTENDED_FEATURES,
    "airport_aware_features": AIRPORT_AWARE_FEATURES,
    "exact_species_policy": (
        "Preserved for descriptive analysis; excluded from core model. "
        "May be tested later only with training-fitted rare-category handling."
    ),
    "seed": SEED
}

with open(DOCS_DIR / "feature_lists.json", "w", encoding="utf-8") as f:
    json.dump(feature_lists, f, indent=2)

print(json.dumps(feature_lists, indent=2))

{
  "primary_target": "INDICATED_DAMAGE",
  "secondary_targets": [
    "DAMAGE_LEVEL"
  ],
  "component_targets": [
    "DAM_RAD",
    "DAM_WINDSHLD",
    "DAM_NOSE",
    "DAM_ENG1",
    "DAM_ENG2",
    "DAM_ENG3",
    "DAM_ENG4",
    "DAM_PROP",
    "DAM_WING_ROT",
    "DAM_FUSE",
    "DAM_LG",
    "DAM_TAIL",
    "DAM_LGHTS",
    "DAM_OTHER"
  ],
  "core_features": [
    "SEASON",
    "MONTH_SIN",
    "MONTH_COS",
    "WILDLIFE_TYPE",
    "SIZE",
    "NUM_STRUCK",
    "AC_CLASS",
    "AC_MASS_GROUP",
    "TYPE_ENG",
    "NUM_ENGS",
    "WARNED"
  ],
  "extended_features": [
    "SEASON",
    "MONTH_SIN",
    "MONTH_COS",
    "WILDLIFE_TYPE",
    "SIZE",
    "NUM_STRUCK",
    "AC_CLASS",
    "AC_MASS_GROUP",
    "TYPE_ENG",
    "NUM_ENGS",
    "WARNED",
    "PHASE_OF_FLIGHT",
    "HEIGHT",
    "SPEED",
    "TIME_OF_DAY",
    "SKY",
    "PRECIPITATION",
    "STATE",
    "FAAREGION"
  ],
  "airport_aware_features": [
    "SEASON",
    "MONTH_SIN",
    "MONTH_COS",
    "WILDLIFE_TYPE",
 

## 15. Chronological validation metadata

The locked benchmark periods are:

- **Training:** 1990–2018
- **Validation:** 2019–2021
- **Final chronological test:** 2022–2024

This split is more defensible than a random split for the main evaluation because future reporting years may differ from earlier years. Later modelling must also run a **recent-window sensitivity analysis**, such as training from 2000 onward, to determine whether very old reporting practices materially affect performance.

A random split may still be used as a benchmark, but it cannot replace chronological validation.

In [19]:
def assign_period(year):
    if 1990 <= year <= 2018:
        return "train_1990_2018"
    if 2019 <= year <= 2021:
        return "validation_2019_2021"
    if 2022 <= year <= 2024:
        return "test_2022_2024"
    return "outside_locked_period"

analysis_df["TEMPORAL_SPLIT"] = analysis_df["INCIDENT_YEAR"].map(assign_period)

validation_summary = (
    analysis_df.groupby("TEMPORAL_SPLIT", dropna=False)
    .agg(
        records=("INDEX_NR", "size"),
        damage_rate=("INDICATED_DAMAGE", "mean"),
        first_year=("INCIDENT_YEAR", "min"),
        last_year=("INCIDENT_YEAR", "max")
    )
    .reset_index()
)
validation_summary["damage_rate"] *= 100

validation_metadata = {
    "train_years": [1990, 2018],
    "validation_years": [2019, 2021],
    "test_years": [2022, 2024],
    "external_years_reserved": [2025],
    "partial_year_excluded": 2026,
    "random_benchmark_seed": SEED,
    "required_sensitivity_analysis": {
        "description": "Repeat temporal evaluation using a more recent training window.",
        "candidate_train_start_year": 2000
    }
}

with open(
    OUTPUT_DIR / "validation_metadata.json", "w", encoding="utf-8"
) as f:
    json.dump(validation_metadata, f, indent=2)

validation_summary.to_csv(
    OUTPUT_DIR / "validation_period_summary.csv", index=False
)
display(validation_summary.round(2))

,TEMPORAL_SPLIT,records,damage_rate,first_year,last_year
0,test_2022_2024,59219,3.79,2022,2024
1,train_1990_2018,215274,7.77,1990,2018
2,validation_2019_2021,44614,4.31,2019,2021


## 16. Airport groups for later held-out validation

Notebook 06 must test generalization to airports not represented in training. Notebook 02 prepares the metadata but does not decide the final held-out airport sample using performance results.

The table below records each airport's training-period support and later-period presence. Rare and unseen airports can then be selected with a reproducible rule.

In [20]:
if "AIRPORT_ID" in analysis_df.columns:
    airport_period_support = (
        analysis_df.pivot_table(
            index="AIRPORT_ID",
            columns="TEMPORAL_SPLIT",
            values="INDEX_NR",
            aggfunc="size",
            fill_value=0
        )
        .reset_index()
    )

    for col in [
        "train_1990_2018", "validation_2019_2021", "test_2022_2024"
    ]:
        if col not in airport_period_support.columns:
            airport_period_support[col] = 0

    airport_period_support["unseen_in_training_flag"] = (
        airport_period_support["train_1990_2018"].eq(0) &
        (
            airport_period_support["validation_2019_2021"] +
            airport_period_support["test_2022_2024"]
        ).gt(0)
    ).astype("Int64")

    airport_period_support.to_csv(
        OUTPUT_DIR / "airport_validation_groups.csv", index=False
    )
    display(
        airport_period_support.sort_values(
            ["unseen_in_training_flag", "test_2022_2024"],
            ascending=False
        ).head(20)
    )

TEMPORAL_SPLIT,AIRPORT_ID,test_2022_2024,train_1990_2018,validation_2019_2021,unseen_in_training_flag
2159,KXWA,34,0,34,1
851,KCFO,33,0,3,1
2260,MHPR,6,0,0,1
292,65NJ,5,0,4,1
2628,VOBL,4,0,0,1
628,KACZ,3,0,0,1
1006,KDYB,3,0,1,1
1626,KNTD,3,0,0,1
2261,MHRO,3,0,0,1
2556,SKPE,3,0,0,1


## 17. Missing-data treatment report

The report records the treatment applied to each field and distinguishes deterministic handling from later learned preprocessing.

Numerical imputation is not performed in this notebook. For example, the median for `HEIGHT` or `SPEED` must be learned from the training data inside a modelling pipeline.

In [21]:
missing_treatment_rows = []

for col in analysis_df.columns:
    raw_col = col[:-4] if col.endswith("_RAW") else col
    missing_pct = round(analysis_df[col].isna().mean() * 100, 2)

    if col in checkbox_fields:
        treatment = "Blank converted to zero after binary-value validation"
        stage = "Deterministic"
    elif col in ["NR_INJURIES", "NR_FATALITIES"]:
        treatment = "Blank converted to zero as no reported cases"
        stage = "Deterministic"
    elif col in categorical_fields:
        treatment = "Preserve explicit Unknown; blank standardized to Not reported"
        stage = "Deterministic"
    elif col in ["HEIGHT", "SPEED", "DISTANCE"]:
        treatment = "Keep missing; training pipeline may impute if approved"
        stage = "Learned preprocessing later"
    elif col in ["IMAGE", "REMAINS_COLLECTED", "REMAINS_SENT", "INGESTED_OTHER"]:
        treatment = "Preserve missing; blank not proven to mean No"
        stage = "Deterministic preservation"
    elif col.endswith("_RAW"):
        treatment = "Preserved unchanged for audit"
        stage = "Audit"
    else:
        treatment = "Preserve unless another documented rule applies"
        stage = "Deterministic preservation"

    missing_treatment_rows.append({
        "field": col,
        "missing_pct_after_deterministic_cleaning": missing_pct,
        "treatment": treatment,
        "stage": stage
    })

missing_treatment = pd.DataFrame(missing_treatment_rows)
missing_treatment.to_csv(
    OUTPUT_DIR / "missing_value_treatment_by_field.csv", index=False
)

display(
    missing_treatment.sort_values(
        "missing_pct_after_deterministic_cleaning", ascending=False
    ).head(30)
)

,field,missing_pct_after_deterministic_cleaning,treatment,stage
9,AIRPORT_LONGITUDE,100.00,Preserve unless another documented rule applies,Deterministic preservation
8,AIRPORT_LATITUDE,100.00,Preserve unless another documented rule applies,Deterministic preservation
37,INCIDENT_LONGITUDE,100.00,Preserve unless another documented rule applies,Deterministic preservation
36,INCIDENT_LATITUDE,100.00,Preserve unless another documented rule applies,Deterministic preservation
83,BIRD_BAND_NUMBER,99.77,Preserve unless another documented rule applies,Deterministic preservation
82,EFFECT_OTHER,99.20,Preserve unless another documented rule applies,Deterministic preservation
31,ENG_4_POS,98.90,Preserve unless another documented rule applies,Deterministic preservation
44,COST_OTHER_INFL_ADJ,98.38,Preserve unless another documented rule applies,Deterministic preservation
42,COST_OTHER,98.38,Preserve unless another documented rule applies,Deterministic preservation
41,COST_REPAIRS,98.35,Preserve unless another documented rule applies,Deterministic preservation


## 18. Export the canonical artifacts

Two datasets are created.

### Full analytical dataset

`faa_strikes_analytical.csv` retains targets, post-event outcomes, component fields, identifiers, and descriptive variables. Notebook 03 should use this file because it needs historical counts and rates.

### Binary modelling dataset

`faa_strikes_binary_model.csv` contains only:

- the primary target,
- approved core, extended, and airport-aware predictors,
- split metadata,
- audit flags needed to exclude or review records.

It excludes `DAMAGE_LEVEL`, all `DAM_*` fields, injuries, fatalities, costs, operational consequences, ingestion outcomes, and other post-event information.

Later modelling notebooks must still construct `X` from `feature_lists.json`; they must not infer predictors from every column present.

In [22]:
full_output_path = PROCESSED_DIR / "faa_strikes_analytical.csv"
binary_output_path = PROCESSED_DIR / "faa_strikes_binary_model.csv"

analysis_df.to_csv(full_output_path, index=False)

binary_allowed = list(dict.fromkeys(
    [PRIMARY_TARGET] +
    CORE_FEATURES +
    EXTENDED_FEATURES +
    AIRPORT_AWARE_FEATURES +
    [
        "TEMPORAL_SPLIT",
        "TARGET_CONFLICT_FLAG",
        "INCIDENT_YEAR",
        "INDEX_NR"
    ]
))

binary_allowed = [c for c in binary_allowed if c in analysis_df.columns]
binary_df = analysis_df[binary_allowed].copy()
binary_df.to_csv(binary_output_path, index=False)

print("Full analytical dataset:")
print(" ", full_output_path)
print(" ", analysis_df.shape)

print("\nRestricted binary-modelling dataset:")
print(" ", binary_output_path)
print(" ", binary_df.shape)

Full analytical dataset:
  c:\Users\davis\Programing\MDA_programs\Capstone\faa-wildlife-strike-damage-analysis\data\processed\faa_strikes_analytical.csv
  (319107, 147)

Restricted binary-modelling dataset:
  c:\Users\davis\Programing\MDA_programs\Capstone\faa-wildlife-strike-damage-analysis\data\processed\faa_strikes_binary_model.csv
  (319107, 25)


### Leakage assertion

The restricted binary file must not contain any secondary target, component outcome, cost, injury, fatality, operational consequence, ingestion, or post-event damage field.

In [ ]:
forbidden_binary_fields = set(SECONDARY_TARGETS + COMPONENT_TARGETS)
forbidden_binary_fields |= {
    c for c in analysis_df.columns
    if c.startswith(("DAM_", "STR_", "ING_"))
}
forbidden_binary_fields |= {
    c for c in [
        "EFFECT", "EFFECT_OTHER", "AOS",
        "COST_REPAIRS", "COST_OTHER",
        "COST_REPAIRS_INFL_ADJ", "COST_OTHER_INFL_ADJ",
        "NR_INJURIES", "NR_FATALITIES",
        "REMAINS_COLLECTED", "REMAINS_SENT", "INGESTED_OTHER"
    ] if c in analysis_df.columns
}

leaked_fields = sorted(set(binary_df.columns) & forbidden_binary_fields)
assert not leaked_fields, (
    "Leakage fields entered the binary modelling export: "
    + ", ".join(leaked_fields)
)

print("Leakage assertion passed.")

Leakage assertion passed.


## 19. Final preparation checkpoint

The checkpoint gives later readers a concise record of what changed and why.

In [25]:
checkpoint = pd.DataFrame([
    ["Source rows", len(df)],
    ["Retained analytical rows", len(analysis_df)],
    ["Retained years", "1990–2024"],
    ["Excluded recent years", "2025 and partial 2026"],
    ["Redundant repeated rows removed", len(removed_repeated_rows)],
    ["Unresolved repeated identifier groups", unresolved_repeated["INDEX_NR"].nunique()],
    ["Target conflicts flagged", int(analysis_df["TARGET_CONFLICT_FLAG"].sum())],
    ["Exact species labels retained", analysis_df["SPECIES"].nunique(dropna=False)],
    ["Broad wildlife groups", analysis_df["WILDLIFE_TYPE"].nunique(dropna=False)],
    ["Core feature count", len(CORE_FEATURES)],
    ["Extended feature count", len(EXTENDED_FEATURES)],
    ["Airport-aware feature count", len(AIRPORT_AWARE_FEATURES)],
    ["Learned preprocessing fitted here", "No"],
], columns=["item", "result"])

checkpoint.to_csv(OUTPUT_DIR / "notebook02_checkpoint.csv", index=False)
display(checkpoint)

,item,result
0,Source rows,348146
1,Retained analytical rows,319107
2,Retained years,1990–2024
3,Excluded recent years,2025 and partial 2026
4,Redundant repeated rows removed,4
5,Unresolved repeated identifier groups,0
6,Target conflicts flagged,0
7,Exact species labels retained,927
8,Broad wildlife groups,5
9,Core feature count,11


## 20. Interpretation and hand-off to Notebook 03

The preparation stage creates one consistent view of the reported-strike data without pretending that uncertain values are known.

- Restricting the main population to 1990–2024 prevents the partial 2026 year from distorting comparisons and preserves 2025 for a stronger later-year check.
- The seven reviewed squirrel pairs are removed only after the synonym mapping makes their redundancy explicit. This rule avoids double-counting while retaining the removed records as evidence.
- Exact species labels remain available for historical summaries, but the main predictor design relies on wildlife type and size. This reduces sparse scenario combinations and should generalize better than hundreds of exact labels.
- `NUM_SEEN` and `NUM_STRUCK` remain ordered categories. Treating their ranges as midpoints would introduce unsupported precision.
- Target disagreements are visible through `TARGET_CONFLICT_FLAG`. Binary modelling may retain them because the primary target is unchanged, whereas severity modelling must omit them until the conflict is resolved.
- Zero is used only where the field's structure supports a “none reported” interpretation. Administrative and reporting blanks are not automatically turned into negative answers.
- Extreme flight values are flagged rather than capped. Later analyses can examine whether results change when flagged observations are excluded.
- The binary-modelling export physically removes known post-event leakage, while the complete analytical file keeps those fields for descriptive, severity, and component analyses.
- Chronological periods are locked before modelling. Later notebooks must compare them with a recent-training-window sensitivity analysis because damage prevalence and reporting completeness vary over time.
- No learned preprocessing was fitted here. This keeps future-year information out of imputation, encoding, scaling, rare-category handling, feature selection, and resampling.

### Notebook 03 should consume

- `data/processed/faa_strikes_analytical.csv`
- `docs/mappings/species_synonym_mapping.csv`
- `docs/mappings/species_to_wildlife_type_review.csv`
- `docs/feature_eligibility.csv`
- `outputs/02_data_preparation/validation_metadata.json`

Notebook 03 should describe historical patterns using counts beside rates, avoid causal language, and avoid interpreting an airport's report count as an ordinary-flight strike probability.

## Manual completion before modelling

The following checks should be completed by a teammate and recorded in the exported files:

1. Review `species_to_wildlife_type_review.csv`, especially labels placed in `Other wildlife`.
2. Confirm that each `STR_*`, `DAM_*`, and `ING_*` field is a checkbox whose blank source value means the box was not selected.
3. Review any unexpected `NUM_SEEN` or `NUM_STRUCK` categories.
4. Review records flagged by numerical plausibility checks rather than deleting them automatically.
5. Peer-review every `pending_review` row in `feature_eligibility.csv`.
6. Run the notebook from a fresh kernel and confirm that all artifacts are regenerated with portable paths.